# <font color="#ff3838ff" face="Palatino Linotype">**PIPELINE RISET: PUPIL DATASET PROCESSOR (PDP)**</font>

<details>
<summary>📌 <b>Lihat Diagram Pipeline Riset SOP (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://cilacap-loki.github.io/pupil-dataset-analysis/assets/img/pipeline_riset.png?raw=true" width="100%" alt="Pipeline Riset SOP">

  
</p>
</details>


## <font color="#38b6ff" face="Palatino Linotype">**FASE 1: DATA PREPARATION & CLEANING**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 1: Data Preparation & Cleaning (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacap-loki/pupil-dataset-analysis/raw/main/assets/img/fase1_visualisasi_garis_besar.png?raw=true" width="100%" alt="FASE 1 Data Preparation & Cleaning">
</p>
</details>

Tahap penyiapan data awal dari video mentah original hingga menjadi kepingan gambar *PNG Lossless* yang terisolasi pada jendela Segmen Optimal (30 Detik), serta pra-pemrosesan pemotongan area mata (ROI) yang stabil.

**Cakupan Standar Operasional Prosedur (SOP):**
* **SOP-01 (Inisialisasi Environment & Konfigurasi Workspace):** Penyiapan sesi komputasi dan pemetaan dataset Google Drive.
* **SOP-02 (Ekstraksi Direct Frame PNG Lossless):** Pencarian rentang Segmen Optimal (30 Detik) (kedipan minimal) dan ekstraksi *direct frame PNG* tanpa kompresi.
* **SOP-03 (Pra-Pemrosesan Grayscale & Stabilized Crop ROI):** Pemotongan area mata dengan fitur *EMA Stabilization* bebas shaking.


In [ ]:
# @title 1️⃣ SOP-01: Inisialisasi Environment & Konfigurasi Workspace
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Hubungkan ke Google Drive
drive.mount('/content/drive')

# 2. Folder Utama tempat HASIL output akan disimpan
ROOT_DIR = '/content/drive/MyDrive/Hibah Penelitian/Outputs'
os.makedirs(ROOT_DIR, exist_ok=True)
print(f"[INFO] Folder penyimpanan utama disiapkan di: {ROOT_DIR}\n")

# 3. Memindai Folder Hibah Penelitian secara Presisi
HIBAH_ROOT = '/content/drive/MyDrive/Hibah Penelitian'
if not os.path.exists(HIBAH_ROOT):
    matches = glob.glob('/content/drive/MyDrive/*[Hh][Ii][Bb][Aa][Hh]*')
    if matches:
        HIBAH_ROOT = matches[0]

folder_map = {}
if os.path.exists(HIBAH_ROOT):
    for root, dirs, files in os.walk(HIBAH_ROOT):
        if 'Hibah Penelitian/Outputs' in root:
            continue
        video_in_dir = [f for f in files if f.lower().endswith(('.mp4', '.avi', '.mkv'))]
        if video_in_dir:
            rel_path = os.path.relpath(root, HIBAH_ROOT)
            label = rel_path if rel_path != '.' else 'Folder Utama Hibah'
            folder_map[label] = root

if not folder_map:
    print("[WARN] Tidak ditemukan folder video di dalam Hibah Penelitian.")
    print("[INFO] Menggunakan path default...")
    global DEBUGGING_MODE
    DEBUGGING_MODE = 'Yes'
    vid_path = '/content/drive/MyDrive/Hibah Penelitian/original/10-06-2026/001.MP4'
    vid_name = os.path.splitext(os.path.basename(vid_path))[0]
    RES_DIR = os.path.join(ROOT_DIR, vid_name)
    dir_raw = os.path.join(RES_DIR, '1_Raw_Frames_SOP01&02')
    dir_gray = os.path.join(RES_DIR, '2_Grayscale_ROI_SOP03')
    dir_mask = os.path.join(RES_DIR, '3_Mask_Pupil_SOP04')
    dir_hasil = os.path.join(RES_DIR, '4_Hasil_Analisis_SOP05_06')
    for d in [dir_raw, dir_gray, dir_mask, dir_hasil]:
        os.makedirs(d, exist_ok=True)
else:
    folder_options = [(lbl, path) for lbl, path in sorted(folder_map.items())]
    
    folder_dropdown = widgets.Dropdown(
        options=folder_options,
        description='1. Tanggal Sesi:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    
    video_dropdown = widgets.Dropdown(
        options=[],
        description='2. Responden:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    

    debugging_dropdown = widgets.Dropdown(
        options=['Yes', 'No'],
        value='Yes',
        description='3. Debug Mode:',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='70%')
    )
    
    def update_debugging_mode(change):
        global DEBUGGING_MODE
        DEBUGGING_MODE = debugging_dropdown.value
        
    debugging_dropdown.observe(update_debugging_mode, names='value')
    
    output_box = widgets.Output()
    
    def update_video_options(change):
        selected_folder = folder_dropdown.value
        vids = []
        if selected_folder and os.path.exists(selected_folder):
            for f in sorted(os.listdir(selected_folder)):
                if f.lower().endswith(('.mp4', '.avi', '.mkv')):
                    vids.append((f, os.path.join(selected_folder, f)))
        video_dropdown.options = vids
        if vids:
            video_dropdown.value = vids[0][1]
            
    def update_selected_video(change):
        global vid_path, vid_name, RES_DIR, dir_raw, dir_gray, dir_mask, dir_hasil
        selected_vid = video_dropdown.value
        if selected_vid:
            vid_path = selected_vid
            vid_name = os.path.splitext(os.path.basename(vid_path))[0]
            RES_DIR = os.path.join(ROOT_DIR, vid_name)
            dir_raw = os.path.join(RES_DIR, '1_Raw_Frames_SOP01&02')
            dir_gray = os.path.join(RES_DIR, '2_Grayscale_ROI_SOP03')
            dir_mask = os.path.join(RES_DIR, '3_Mask_Pupil_SOP04')
            dir_hasil = os.path.join(RES_DIR, '4_Hasil_Analisis_SOP05_06')
            
            for d in [dir_raw, dir_gray, dir_mask, dir_hasil]:
                os.makedirs(d, exist_ok=True)
                
            with output_box:
                clear_output()
                print(f"======================================================================")
                print(f"[RESPONDEN AKTIF: {vid_name}]")
                print(f"======================================================================")
                print(f"🎯 Video Terpilih : {vid_path}")
                print(f"📁 Folder Output  : {RES_DIR}")
                print(f"🔗 Google Drive   : https://drive.google.com/drive/my-drive")
                print(f"✅ Workspace siap diproses pada SOP-01 & SOP-02!\n")
                
    folder_dropdown.observe(update_video_options, names='value')
    video_dropdown.observe(update_selected_video, names='value')
    
    print("📌 PILIH DATASET TERFOKUS (HIBAH PENELITIAN):\n")
    display(folder_dropdown)
    display(video_dropdown)
    display(debugging_dropdown)
    display(output_box)
    
    update_video_options(None)
    update_selected_video(None)
    update_debugging_mode(None)



In [ ]:
# @title 2️⃣ SOP-02: Ekstraksi Otomatis 30 Detik Segmen Stabil

def check_blink_fast(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(blurred)
    cx, cy = min_loc
    
    roi_size = 800
    h, w = gray.shape
    x_min, y_min = max(0, cx - roi_size // 2), max(0, cy - roi_size // 2)
    x_max, y_max = min(w, cx + roi_size // 2), min(h, cy + roi_size // 2)
    roi_blurred = blurred[y_min:y_max, x_min:x_max]
    
    thresh_val = min(min_val + 28, 130)
    _, thresh = cv2.threshold(roi_blurred, thresh_val, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    radius = 0
    if contours:
        valid_contours = [c for c in contours if cv2.contourArea(c) > 100]
        if valid_contours:
            best_contour = max(valid_contours, key=cv2.contourArea)
            _, radius = cv2.minEnclosingCircle(best_contour)
    return radius * 2

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-01 & SOP-02: EKSTRAKSI 30 DETIK EMAS")
print(f"======================================================================")
print(f"[STEP 1/2] Memindai kedipan minimal pada video: {vid_path}")

cap = cv2.VideoCapture(vid_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
target_frames = 1500

if total_frames <= target_frames:
    start_f = 0
    end_f = total_frames
else:
    blinks = []
    frame_idx = 0
    from tqdm.notebook import tqdm
    with tqdm(total=total_frames//5, desc=f"Scanning [{vid_name}]") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: 
                break
            if frame_idx % 5 == 0:
                if check_blink_fast(frame) < 5:
                    blinks.append(frame_idx)
                pbar.update(1)
            frame_idx += 1
            
    best_start = 0
    min_blinks = float('inf')
    for start_i in range(0, total_frames - target_frames, 50):
        end_i = start_i + target_frames
        blink_count = sum(1 for b in blinks if start_i <= b <= end_i)
        if blink_count < min_blinks:
            min_blinks = blink_count
            best_start = start_i
            
    start_f = best_start
    end_f = best_start + target_frames

# --- LOGIKA DEBUG BERADA DI LUAR IF-ELSE ---
if globals().get('DEBUGGING_MODE', 'No') == 'Yes':
    end_f = start_f + min(300, target_frames)
    print(f"[DEBUG MODE] Memotong eksekusi menjadi {end_f - start_f} frame saja.")
# -------------------------------------------

print(f"
[STEP 2/2] Rentang Segmen Optimal ditemukan pada Frame {start_f} sampai {end_f}.")
print(f"Mengekstrak Frame PNG Lossless langsung ke folder: {dir_raw}")

cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
frames_to_read = end_f - start_f

for i in tqdm(range(frames_to_read), desc=f"Extracting [{vid_name}]"):
    ret, frame = cap.read()
    if not ret: 
        break
    raw_name = f"{vid_name}_frame_{i:04d}.png"
    cv2.imwrite(os.path.join(dir_raw, raw_name), frame)

cap.release()
print(f"
======================================================================")
print(f"📁 Nama Folder Drive : 1_Raw_Frames_SOP01&02")
print(f"📍 Drive Local Path  : {dir_raw}")
print(f"🔗 Akses Google Drive: https://drive.google.com/drive/my-drive")
print(f"======================================================================")


In [ ]:
# @title 3️⃣ SOP-03: Pra-Pemrosesan Grayscale & Stabilized Crop ROI 800x800
import os, cv2
import numpy as np
from tqdm.notebook import tqdm

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-03: PRA-PEMROSESAN (GRAYSCALE & STABILIZED CROP ROI)")
print(f"======================================================================")

raw_files = sorted(os.listdir(dir_raw))
print(f"[INFO] Memproses {len(raw_files)} frame ke 2_Grayscale_ROI_SOP03 (Stabilized EMA)...")

roi_size = 800
cx_smooth, cy_smooth = None, None
alpha = 0.08  # EMA smoothing factor untuk gerakan stabil tanpa shaking

for filename in tqdm(raw_files, desc=f"Grayscale & ROI [{vid_name}]"):
    frame = cv2.imread(os.path.join(dir_raw, filename))
    if frame is None: continue
    
    H, W = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 1. BORDER MARGIN GATING (Mengabaikan 15% tepi terluar frame agar hijab/pakaian tidak mengganggu)
    margin_x = int(W * 0.15)
    margin_y = int(H * 0.15)
    roi_active = gray[margin_y:H-margin_y, margin_x:W-margin_x]
    
    # 2. MEDIAN BLUR MINLOC: Menemukan Titik Pusat Pupil Presisi Tanpa Distorsi Kontur
    mblur_active = cv2.medianBlur(roi_active, 25)
    min_v, _, min_loc_roi, _ = cv2.minMaxLoc(mblur_active)
    
    target_x = margin_x + min_loc_roi[0]
    target_y = margin_y + min_loc_roi[1]
    
    # 3. EMA Smoothing untuk Kestabilan ROI tanpa Shaking
    if cx_smooth is None or cy_smooth is None:
        cx_smooth, cy_smooth = float(target_x), float(target_y)
    else:
        cx_smooth = (alpha * target_x) + ((1 - alpha) * cx_smooth)
        cy_smooth = (alpha * target_y) + ((1 - alpha) * cy_smooth)
            
    # 4. Kalkulasi & Pemotongan Bounding Box ROI 800x800
    x1 = int(cx_smooth - roi_size / 2)
    y1 = int(cy_smooth - roi_size / 2)
    x2 = x1 + roi_size
    y2 = y1 + roi_size
    
    pad_top, pad_bottom, pad_left, pad_right = 0, 0, 0, 0
    if y1 < 0: pad_top = -y1; y1 = 0
    if y2 > H: pad_bottom = y2 - H; y2 = H
    if x1 < 0: pad_left = -x1; x1 = 0
    if x2 > W: pad_right = x2 - W; x2 = W
    
    roi = gray[y1:y2, x1:x2]
    if pad_top > 0 or pad_bottom > 0 or pad_left > 0 or pad_right > 0:
        roi = cv2.copyMakeBorder(roi, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REPLICATE)
        
    # Simpan sebagai Grayscale standar murni tanpa CLAHE
    cv2.imwrite(os.path.join(dir_gray, filename), roi)

print("[INFO] Tahap SOP-03 Selesai (Grayscale & Motion-Stabilized Crop Active).")



## <font color="#38b6ff" face="Palatino Linotype">**FASE 2: PENGEMBANGAN & DEBUGGING PIPELINE OTOMASI**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 2: Pengembangan & Debugging Pipeline Otomasi (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacap-loki/pupil-dataset-analysis/blob/main/assets/img/fase2_visualisasi_garis_besar.png?raw=true" width="100%" alt="FASE 2 Pengembangan & Debugging Pipeline Otomasi">
</p>
</details>

Tahap segmentasi pupil, ekstraksi fitur osilasi hippus temporal, dan analisis medis tingkat lanjut.

**Cakupan Standar Operasional Prosedur (SOP):**
* **SOP-04 (Segmentasi Mask Pupil & Video Verifikasi):** Segmentasi biner pupil, *Anatomical Reconstruction*, dan rendering 3 video MP4 real-time.
* **SOP-05 (Tracking Pupil & Analisis Sinyal):** Ekstraksi fitur medis hippus, ekspor berkas CSV, dan rendering grafik PNG.
* **SOP-06 (Generator Laporan PDF Klinis):** Penyusunan dan penertiban dokumen cetak PDF Diagnostik Klinis Responden.


In [ ]:
# @title 4️⃣1️⃣ SOP-04 (Langkah 1): Inisialisasi Universal V3 Engine (Konstanta, Core, Kalibrasi)
import cv2
import json
import math
import numpy as np
import os
import sys


# ─────────────────────────────────────────────────────────
#  KONFIGURASI
# ─────────────────────────────────────────────────────────

STRATEGIES = {
    # glint_removal : inpaint pantulan kuat sebelum sweep
    # dark_margin   : inside_mean blob boleh lebih terang dari anchor_val sejauh ini
    # dia_floor     : diameter minimum yang dianggap "pupil penuh" (aturan potong aktif di atasnya)
    "standard":   {"glint_removal": False, "dark_margin": 18.0, "dia_floor": 45.0},
    "compact":    {"glint_removal": False, "dark_margin": 15.0, "dia_floor": 34.0},
    "reflective": {"glint_removal": True,  "dark_margin": 18.0, "dia_floor": 45.0},
}

CALIBRATION_FRAMES   = 60     # jumlah frame sampel untuk kalibrasi
MIN_CONFIDENCE       = 0.55   # threshold minimum confidence untuk tidak FALLBACK

N_ANCHORS_CALIB      = 4      # jumlah minima lokal saat kalibrasi / re-akuisisi
N_ANCHORS_TRACK      = 2      # minima tambahan saat tracking (anchor prediksi tetap pertama)
ANCHOR_MASK_RADIUS   = 90     # jarak minimum antar minima lokal

ROI_SWEEP            = 300    # ukuran ROI sweep di sekitar anchor
OFFSET_LO            = 6      # sweep threshold: anchor_val + 6
OFFSET_HI            = 87     #         ... sampai anchor_val + 86
OFFSET_STEP          = 2
DIA_ABS_MIN          = 36.0   # clamp fisiologis absolut (skala dataset ini)
DIA_ABS_MAX          = 170.0
DIA_MIN_RATIO        = 0.50   # gerbang tracking: dia_min = ratio x baseline
DIA_MAX_RATIO        = 1.80
MAX_CENTER_SHIFT_TRACK = 60.0
NO_DET_STREAK_RESET  = 10     # frame NO_DETECTION berturut sebelum re-akuisisi penuh

BASELINE_SANITY_LO   = 45.0   # baseline kalibrasi harus dalam rentang ini
BASELINE_SANITY_HI   = 140.0

GLOBAL_INSIDE_MARGIN = 25.0
INSIDE_IQR_MAX     = 30.0   # IQR intensitas di dalam kandidat: blob yang mencakup
                            # iris (bimodal pupil+iris) ditolak; pupil asli unimodal.
                            # pupil dengan sisa gradien glint (lpw) bisa sampai ~28.   # inside_mean blob wajib <= min global frame + margin ini.
                              # Membunuh blob raksasa iris+sklera: kontras cincinnya
                              # besar (sklera terang) tapi isinya tidak gelap-pupil.

W_ASPECT   = 30.0             # skor zona: kebulatan bentuk
W_CONTRAST = 1.5              # skor zona: kontras pupil-iris
W_EXPECT   = 0.8              # skor zona: kedekatan diameter ke ekspektasi
                              # (baseline kalibrasi / diameter frame sebelumnya)

# Blink detection (dua-kondisi; gate mean kini adaptif dari kalibrasi)
BLINK_DARK_THR       = 170
BLINK_MAX_OPENING_PX = 110
BLINK_MEAN_MARGIN    = 18.0   # gate = median roi_mean mata-terbuka + margin ini
BLINK_ABS_MEAN_THR   = 215.0  # fallback absolut (dipakai pra-kalibrasi)
BLINK_ANCHOR_THR     = 150


# ─────────────────────────────────────────────────────────
#  TRACKER TEMPORAL (tidak berubah dari v2)
# ─────────────────────────────────────────────────────────

class PupilTemporalTracker:
    def __init__(self, alpha=0.35, max_dev_ratio=0.08, history_len=5, aspect_scale=1.0):
        self.alpha          = alpha
        self.max_dev_ratio  = max_dev_ratio
        self.history_len    = history_len
        self.aspect_scale   = aspect_scale
        self.smoothed_center   = None
        self.smoothed_diameter = None
        self.smoothed_axes     = None
        self.smoothed_angle    = None
        self.diameter_history  = []

    def reset(self):
        self.smoothed_center   = None
        self.smoothed_diameter = None
        self.smoothed_axes     = None
        self.smoothed_angle    = None
        self.diameter_history.clear()

    @staticmethod
    def _unwrap_ellipse_angle(angle, reference):
        """Normalisasi sudut periodik 180°: 0° dan 180° adalah orientasi sama."""
        return reference + ((angle - reference + 90.0) % 180.0 - 90.0)

    def update(self, raw_center, raw_axes, raw_angle, raw_diameter):
        if self.aspect_scale > 1.0:
            if raw_axes[0] < raw_axes[1]:
                raw_axes = (raw_axes[0] * self.aspect_scale, raw_axes[1])
            else:
                raw_axes = (raw_axes[0], raw_axes[1] * self.aspect_scale)
            raw_diameter = (raw_axes[0] + raw_axes[1]) / 2.0

        if self.smoothed_diameter is None:
            self.smoothed_center   = raw_center
            self.smoothed_axes     = raw_axes
            self.smoothed_angle    = raw_angle
            self.smoothed_diameter = raw_diameter
        else:
            if len(self.diameter_history) >= 3:
                median_dia = float(np.median(self.diameter_history))
                diff_ratio = abs(raw_diameter - median_dia) / (median_dia + 1e-9)
                if diff_ratio > self.max_dev_ratio:
                    clamped_dia  = median_dia * (1.0 + np.sign(raw_diameter - median_dia) * self.max_dev_ratio)
                    scale_factor = clamped_dia / raw_diameter if raw_diameter > 0 else 1.0
                    raw_diameter = clamped_dia
                    raw_axes     = (raw_axes[0] * scale_factor, raw_axes[1] * scale_factor)
            raw_angle = self._unwrap_ellipse_angle(raw_angle, self.smoothed_angle)
            a = self.alpha
            self.smoothed_center = (
                a * raw_center[0] + (1 - a) * self.smoothed_center[0],
                a * raw_center[1] + (1 - a) * self.smoothed_center[1],
            )
            self.smoothed_diameter = a * raw_diameter + (1 - a) * self.smoothed_diameter
            self.smoothed_axes = (
                a * raw_axes[0] + (1 - a) * self.smoothed_axes[0],
                a * raw_axes[1] + (1 - a) * self.smoothed_axes[1],
            )
            self.smoothed_angle = a * raw_angle + (1 - a) * self.smoothed_angle

        self.diameter_history.append(self.smoothed_diameter)
        if len(self.diameter_history) > self.history_len:
            self.diameter_history.pop(0)

        render_angle = self.smoothed_angle % 180.0
        return {
            "center":   self.smoothed_center,
            "diameter": self.smoothed_diameter,
            "axes":     self.smoothed_axes,
            "angle":    render_angle,
            "ellipse":  (self.smoothed_center, self.smoothed_axes, render_angle),
        }


# ─────────────────────────────────────────────────────────
#  UTILITAS BLINK, KONTRAS, GLINT
# ─────────────────────────────────────────────────────────

def measure_blink_metrics(img):
    """Ukur eye_opening_px dan eye_roi_mean dari box tengah frame."""
    h, w = img.shape[:2]
    y0, y1 = int(h * 0.3125), int(h * 0.6875)
    x0, x1 = int(w * 0.3125), int(w * 0.6875)
    roi = img[y0:y1, x0:x1]
    dark_y, _ = np.where(roi < BLINK_DARK_THR)
    opening = int(dark_y.max() - dark_y.min() + 1) if len(dark_y) else 0
    mean_intensity = float(np.mean(roi))
    return {"eye_opening_px": opening, "eye_roi_mean": mean_intensity}


def is_blink(metrics, anchor_val, mean_gate=BLINK_ABS_MEAN_THR):
    """
    Blink = (eye_opening <= threshold AND eye_roi_mean >= gate adaptif)
    ATAU anchor_val > threshold absolut.
    """
    two_cond = (
        metrics["eye_opening_px"] <= BLINK_MAX_OPENING_PX
        and metrics["eye_roi_mean"] >= mean_gate
    )
    return two_cond or (anchor_val > BLINK_ANCHOR_THR)


def find_pupil_anchor(img):
    """(min_val, min_loc) dari gambar blur 21x21."""
    blurred = cv2.GaussianBlur(img, (21, 21), 0)
    min_val, _, min_loc, _ = cv2.minMaxLoc(blurred)
    return float(min_val), min_loc


def local_dark_minima(blur_img, k=N_ANCHORS_CALIB, mask_r=ANCHOR_MASK_RADIUS):
    """
    [Perbaikan 1] K minima lokal terpisah >= mask_r px pada gambar blur.
    Pupil hampir selalu punya minimum lokalnya sendiri; alis/bulu mata tidak
    bisa membajak satu-satunya anchor lagi.
    Kembalikan list [(val, (x, y)), ...] terurut dari tergelap.
    """
    work = blur_img.astype(np.float64)
    out = []
    for _ in range(k):
        idx = int(np.argmin(work))
        y, x = divmod(idx, work.shape[1])
        out.append((float(blur_img[y, x]), (int(x), int(y))))
        cv2.circle(work, (x, y), mask_r, 1e9, -1)
    return out


def ellipse_contrast(img, center, axes, angle, win=None):
    """
    Kontras pupil-iris dengan dua cincin annular.
    Ring dalam (0..0.65r): inti pupil. Ring luar (1.00..1.18r): iris.
    Jika win=(x0,y0,x1,y1) diberikan, perhitungan dibatasi pada window tsb (cepat).
    Kembalikan (contrast, inside_median, inside_iqr).
    """
    if win is None:
        y0, y1, x0, x1 = 0, img.shape[0], 0, img.shape[1]
    else:
        x0, y0, x1, y1 = win
    sub = img[y0:y1, x0:x1]
    yy, xx = np.ogrid[y0:y1, x0:x1]
    theta = np.deg2rad(angle)
    dx = xx - center[0]
    dy = yy - center[1]
    xr = dx * np.cos(theta) + dy * np.sin(theta)
    yr = -dx * np.sin(theta) + dy * np.cos(theta)
    rx = max(axes[0] / 2.0, 1.0)
    ry = max(axes[1] / 2.0, 1.0)
    radius = np.sqrt((xr / rx) ** 2 + (yr / ry) ** 2)
    inside = sub[radius <= 0.65]
    ring   = sub[(radius >= 1.00) & (radius <= 1.18)]
    if len(inside) < 30 or len(ring) < 30:
        return -np.inf, np.inf
    # inside memakai MEDIAN (robust thd lubang glint terang di tengah pupil,
    # mis. lpw); ring tetap mean. Kontras = ring_mean - inside_median.
    # IQR inside mengukur keseragaman isi: pupil asli ~unimodal gelap;
    # blob yang menyertakan iris ~bimodal (IQR besar).
    inside_med = float(np.median(inside))
    inside_iqr = float(np.percentile(inside, 75) - np.percentile(inside, 25))
    return float(np.mean(ring) - inside_med), inside_med, inside_iqr, float(np.mean(ring))


def preprocess_glint(img):
    """Hilangkan glint (pantulan kuat) dengan inpaint — strategi reflective.
    Syarat dua-ambang: piksel sangat terang (>150) di area gelap (blur<140);
    dilatasi 2x agar halo gradien di sekitar inti glint ikut ter-inpaint."""
    blurred15  = cv2.GaussianBlur(img, (15, 15), 0)
    glint_mask = np.zeros_like(img, dtype=np.uint8)
    glint_mask[(img > 150) & (blurred15 < 140)] = 255
    kernel_g   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    glint_mask = cv2.dilate(glint_mask, kernel_g, iterations=2)
    return cv2.inpaint(img, glint_mask, 3, cv2.INPAINT_TELEA)


# ─────────────────────────────────────────────────────────
#  DETEKSI INTI v3: SWEEP OFFSET + ZONA STABIL
# ─────────────────────────────────────────────────────────

# ==========================================
#  BAGIAN 2: ENGINE DETEKSI CORE V3
# ==========================================
def _sweep_rows(img_proc, a_val, a_xy):
    """
    Sweep threshold relatif terhadap anchor_val di ROI sekitar anchor.
    Per offset, kandidat kontur terbaik dipilih dari bentuk (aspect & jarak),
    lalu diukur kontras/inside-nya. Kembalikan seri kandidat per offset.
    """
    ax, ay = a_xy
    h, w = img_proc.shape[:2]
    half = ROI_SWEEP // 2
    x0 = max(0, ax - half); y0 = max(0, ay - half)
    x1 = min(w, x0 + ROI_SWEEP); y1 = min(h, y0 + ROI_SWEEP)
    x0 = max(0, x1 - ROI_SWEEP); y0 = max(0, y1 - ROI_SWEEP)
    roi = img_proc[y0:y1, x0:x1]

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    rows = []
    for off in range(OFFSET_LO, OFFSET_HI, OFFSET_STEP):
        t = min(255, int(a_val) + off)
        b = cv2.threshold(roi, t, 255, cv2.THRESH_BINARY_INV)[1]
        b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k_close)
        b = cv2.morphologyEx(b, cv2.MORPH_OPEN, k_open)
        contours, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        best = None
        for c in contours:
            if len(c) < 5:
                continue
            area = cv2.contourArea(c)
            if not 800 <= area <= 30000:
                continue
            (cx, cy), axes, ang = cv2.fitEllipse(c)
            minor, major = sorted(axes)
            aspect = minor / major if major else 0.0
            if aspect < 0.45:
                continue
            center = (cx + x0, cy + y0)
            dist = math.hypot(center[0] - ax, center[1] - ay)
            if dist > 55.0:
                continue
            # kontras dihitung untuk TIAP kandidat pada citra yang di-threshold
            # (img_proc): pada strategi reflective lubang glint sudah di-inpaint
            # sehingga inside seragam (IQR kecil) — mengukur citra asli akan
            # menolak pupil berglint sah.
            contrast, mean_inside, inside_iqr, ring_mean = ellipse_contrast(
                img_proc, center, axes, ang, win=(x0, y0, x1, y1))
            if contrast == -np.inf:
                continue
            if inside_iqr > INSIDE_IQR_MAX:
                continue  # blob bimodal (pupil+iris) — bukan pupil murni
            cand = {
                "off": off, "threshold": float(min(255, int(a_val) + off)),
                "dia": (axes[0] + axes[1]) / 2.0, "axes": axes, "angle": ang,
                "center": center, "aspect": aspect, "contrast": contrast,
                "inside": mean_inside, "inside_iqr": inside_iqr,
                "ring_mean": ring_mean, "dist_anchor": dist,
            }
            if best is None or contrast > best["contrast"]:
                best = cand
        if best is not None:
            rows.append(best)
    return rows


def _stable_zone(rows, a_val, dark_margin, dia_floor):
    """
    [Perbaikan 2] Potong seri pada titik blob mulai "memakan" iris:
      c1: inside_mean kandidat > anchor_val + dark_margin
      c2: diameter melonjak > 1.45x prev (atau +32 px) per langkah
      c3: inside naik > 8 bersamaan dengan diameter naik > 8%
      c4: ring_mean kandidat > referensi iris zona + 25 (cincin menyentuh
          sklera; referensi = median ring baris terbentuk pertama zona ini)
    Aturan c2/c3 hanya aktif untuk blob TERBENTUK (prev dia >= dia_floor DAN
    prev aspect >= 0.55): fragmen inti (kecil/pepih) bebas tumbuh menuju pupil
    penuh tanpa memicu pemotongan.
    """
    cut = len(rows)
    ring_refs = []  # ring_mean baris terbentuk pertama (referensi tingkat iris)
    for i in range(1, len(rows)):
        p, c = rows[i - 1], rows[i]
        if c["inside"] > a_val + dark_margin:
            cut = i
            break
        if ring_refs and c["ring_mean"] > (float(np.median(ring_refs)) + 25.0):
            cut = i  # c4: cincin sudah jauh lebih terang dari iris ~ sklera
            break
        if p["dia"] >= dia_floor and p["aspect"] >= 0.55:
            if c["dia"] > 1.45 * p["dia"] or (c["dia"] - p["dia"]) > 32.0:
                cut = i
                break
            if c["inside"] - p["inside"] > 8.0 and c["dia"] > 1.08 * p["dia"]:
                cut = i
                break
        # perbarui referensi iris dari maksimal 3 baris terbentuk pertama
        if len(ring_refs) < 3 and p["dia"] >= dia_floor and p["aspect"] >= 0.55:
            ring_refs.append(p["ring_mean"])
    return rows[:cut]


def _zone_pick(zone, expected_dia=None):
    """[Perbaikan 3] Pilih kandidat zona: aspect + kontras - penalti deviasi ekspektasi."""
    def key(r):
        s = r["aspect"] * W_ASPECT + r["contrast"] * W_CONTRAST
        if expected_dia:
            s -= W_EXPECT * abs(r["dia"] - expected_dia)
        return s
    return max(zone, key=key)


def detect_core(img, cfg, anchors, expected_dia=None, track_center=None,
                global_min=None):
    """
    Deteksi pupil multi-anchor. Untuk setiap anchor: sweep + zona stabil + pick.
    global_min: nilai blur minimum seluruh frame — acuan gerbang gelap absolut
    (blob sah harus gelap seperti pupil, bukan abu-abu seperti iris+sklera).
    Kembalikan (solusi_terbaik | None, daftar_semua_solusi).
    """
    if global_min is None:
        global_min = min((a[0] for a in anchors), default=0.0)
    inside_gate = global_min + GLOBAL_INSIDE_MARGIN

    img_proc = preprocess_glint(img) if cfg["glint_removal"] else img
    solutions = []
    for a_val, a_xy in anchors:
        rows = _sweep_rows(img_proc, a_val, a_xy)
        zone = _stable_zone(rows, a_val, cfg["dark_margin"], cfg["dia_floor"])
        zone = [r for r in zone
                if r["dia"] >= cfg["dia_floor"] * 0.85
                and r["inside"] <= inside_gate]
        if not zone:
            continue
        pick = _zone_pick(zone, expected_dia)
        pick = dict(pick)
        pick["anchor"] = a_xy
        pick["anchor_val"] = a_val
        pick["zone_len"] = len(zone)
        solutions.append(pick)

    if not solutions:
        return None, []

    def g(s):
        sc = s["aspect"] * W_ASPECT + s["contrast"] * W_CONTRAST
        if expected_dia:
            sc -= W_EXPECT * abs(s["dia"] - expected_dia)
        if track_center:
            sc -= 0.2 * math.hypot(s["center"][0] - track_center[0],
                                   s["center"][1] - track_center[1])
        return sc

    best = max(solutions, key=g)
    best["candidate_score"] = g(best)
    return best, solutions


def build_track_anchors(blur_img, last_center, k_extra=N_ANCHORS_TRACK):
    """Anchor tracking: posisi last_center lebih dulu, lalu K minima lokal.
    Nilai anchor prediksi diambil sebagai minimum blur dalam cakram 12 px
    (posisi prediksi bisa bergeser sedikit dari inti pupil)."""
    anchors = []
    if last_center is not None:
        cx, cy = int(round(last_center[0])), int(round(last_center[1]))
        cy = min(max(cy, 0), blur_img.shape[0] - 1)
        cx = min(max(cx, 0), blur_img.shape[1] - 1)
        y0, y1 = max(0, cy - 12), min(blur_img.shape[0], cy + 13)
        x0, x1 = max(0, cx - 12), min(blur_img.shape[1], cx + 13)
        a_val = float(np.min(blur_img[y0:y1, x0:x1]))
        anchors.append((a_val, (cx, cy)))
    for a_val, a_xy in local_dark_minima(blur_img, k=k_extra):
        if all(math.hypot(a_xy[0] - p[0], a_xy[1] - p[1]) > 40 for _, p in anchors):
            anchors.append((a_val, a_xy))
    return anchors


# ─────────────────────────────────────────────────────────
#  KALIBRASI OTOMATIS (v3)
# ─────────────────────────────────────────────────────────

# ==========================================
#  BAGIAN 3: MODUL KALIBRASI OTOMATIS V3
# ==========================================
def run_calibration(input_dir, files):
    """
    [Perbaikan 4] Evaluasi tiap strategi pada 60 frame sampel dengan detect_core.
    Skor = solve_frac + kontras + reproduksibilitas (split even/odd) + konsistensi.
    Sanity gate keras: baseline harus dalam [BASELINE_SANITY_LO, _HI].
    Kembalikan dict hasil kalibrasi lengkap.
    """
    n = len(files)
    calib_n = min(n, 300) # Kalibrasi hanya pada 300 frame pertama untuk konsistensi dengan Debug Mode
    indices = [int(i * calib_n / CALIBRATION_FRAMES) for i in range(CALIBRATION_FRAMES)]
    indices = sorted(set(min(i, n - 1) for i in indices))

    # Tahap 1: kumpulkan SEMUA solusi kandidat per frame per strategi
    strategy_frame_solutions = {s: [] for s in STRATEGIES}
    open_roi_means = []

    for idx in indices:
        img = cv2.imread(os.path.join(input_dir, files[idx]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        blur = cv2.GaussianBlur(img, (21, 21), 0)
        anchor_val = float(cv2.minMaxLoc(blur)[0])
        blink_m = measure_blink_metrics(img)
        if is_blink(blink_m, anchor_val):          # gerbang absolut pra-kalibrasi
            continue
        open_roi_means.append(blink_m["eye_roi_mean"])

        anchors = local_dark_minima(blur, k=N_ANCHORS_CALIB)
        for sname, cfg in STRATEGIES.items():
            _, solutions = detect_core(img, cfg, anchors)
            if solutions:
                strategy_frame_solutions[sname].append(solutions)

    # Tahap 2: pilih ulang dengan expected_dia = baseline tahap-1.
    # Stabilisasi two-pass: fragmen inti (kecil) kalah melawan pupil penuh
    # begitu ada referensi skala; tanpa ini fragmen menang secara sporadis.
    strategy_solutions = {}
    for sname, frame_sols in strategy_frame_solutions.items():
        all_s1 = [s for sols in frame_sols for s in sols]
        if not all_s1:
            strategy_solutions[sname] = []
            continue
        base0 = float(np.median([s["dia"] for s in all_s1]))
        picked = []
        for sols in frame_sols:
            p = max(sols, key=lambda s: s["aspect"] * W_ASPECT
                    + s["contrast"] * W_CONTRAST
                    - W_EXPECT * abs(s["dia"] - base0))
            picked.append(p)
        strategy_solutions[sname] = picked

    scores = {}
    details = {}
    for sname, sols in strategy_solutions.items():
        if len(sols) < 5:
            scores[sname] = 0.0
            details[sname] = {"n_solved": len(sols)}
            continue
        dias = [s["dia"] for s in sols]
        baseline = float(np.median(dias))
        even = dias[0::2]
        odd  = dias[1::2]
        m_e, m_o = float(np.median(even)), float(np.median(odd))
        repro = abs(m_e - m_o) / ((m_e + m_o) / 2.0 + 1e-9)
        solve_frac = len(sols) / max(1, len(indices))
        contrast_med = float(np.median([s["contrast"] for s in sols]))
        dia_cv = float(np.std(dias) / (baseline + 1e-9))
        dia_cons = max(0.0, 1.0 - dia_cv)
        # margin gelap adaptif: seberapa jauh inside_mean pupil asli berada di
        # atas anchor_val (blur-min) pada dataset ini (lpw ~21, 040 ~15)
        inside_offs = [s["inside"] - s["anchor_val"] for s in sols
                       if s.get("anchor_val") is not None]
        dark_margin_est = float(np.clip(np.percentile(inside_offs, 75) + 6.0, 14.0, 30.0)) \
            if inside_offs else 18.0
        score = (0.40 * solve_frac
                 + 0.20 * min(contrast_med, 60.0) / 60.0
                 + 0.25 * max(0.0, 1.0 - 2.0 * repro)
                 + 0.15 * dia_cons)
        scores[sname] = round(score, 4)
        details[sname] = {
            "n_solved": len(sols), "solve_frac": round(solve_frac, 4),
            "baseline": round(baseline, 3), "repro": round(repro, 4),
            "contrast_med": round(contrast_med, 3), "dia_cv": round(dia_cv, 4),
            "dark_margin_est": round(dark_margin_est, 2),
        }

    best_strategy = max(scores, key=lambda s: scores[s])
    best_score = scores[best_strategy]
    baseline = details.get(best_strategy, {}).get("baseline", 0.0)

    # [Perbaikan 4] sanity gate keras
    sane = (best_score >= MIN_CONFIDENCE
            and BASELINE_SANITY_LO <= baseline <= BASELINE_SANITY_HI)

    if sane:
        cal_mode = "AUTO"
    else:
        cal_mode = "FALLBACK"
        pooled = [s["dia"] for sols in strategy_solutions.values() for s in sols]
        baseline = float(np.median(pooled)) if pooled else 70.0
        best_strategy = "standard"

    dia_min = float(min(max(baseline * DIA_MIN_RATIO, DIA_ABS_MIN), DIA_ABS_MAX))
    dia_max = float(min(max(baseline * DIA_MAX_RATIO, DIA_ABS_MIN), DIA_ABS_MAX))

    # [Perbaikan 5] blink gate adaptif
    if open_roi_means:
        blink_mean_gate = float(np.median(open_roi_means)) + BLINK_MEAN_MARGIN
    else:
        blink_mean_gate = BLINK_ABS_MEAN_THR

    dark_margin = details.get(best_strategy, {}).get("dark_margin_est", 18.0)

    return {
        "strategy":          best_strategy,
        "scores":            scores,
        "details":           details,
        "confidence":        float(best_score if sane else best_score),
        "mode":              cal_mode,
        "baseline_dia":      float(baseline),
        "dia_min":           dia_min,
        "dia_max":           dia_max,
        "blink_mean_gate":   blink_mean_gate,
        "dark_margin":       float(dark_margin),
        "baseline_sane":     bool(sane),
    }


# ─────────────────────────────────────────────────────────
#  RENDER DEBUG (untuk review visual)
# ─────────────────────────────────────────────────────────


In [ ]:
# @title 4️⃣2️⃣ SOP-04 (Langkah 2): Kalibrasi & Segmentasi Masker Biner PNG (Universal V3)
import os, cv2
import numpy as np
import math
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-04 (LANGKAH 2): SEGMENTASI MASK PUPIL V3")
print(f"======================================================================")

sub_mask = os.path.join(dir_mask, "1_Mask_Biner")
sub_overlay = os.path.join(dir_mask, "2_Overlay_Pupil")
os.makedirs(sub_mask, exist_ok=True)
os.makedirs(sub_overlay, exist_ok=True)

diameters = []
centers = []
angles = []
ellipses = []
masks_in_memory = []
gray_frames_in_memory = []
gray_files = sorted(os.listdir(dir_gray))

print(f"[INFO] Menjalankan Kalibrasi Otomatis V3 pada {len(gray_files)} frame...")

# --- Fase Kalibrasi ---
full_paths = [os.path.join(dir_gray, f) for f in gray_files]
calib_data = run_calibration(dir_gray, gray_files)

cfg = STRATEGIES[calib_data["strategy"]]
dia_min = calib_data["dia_min"]
dia_max = calib_data["dia_max"]
blink_mean_gate = calib_data["blink_mean_gate"]
conf = calib_data["confidence"]
baseline = calib_data["baseline_dia"]
print(f"   => Strategi Terpilih: {calib_data['strategy'].upper()} (Conf: {conf:.3f}, Baseline: {baseline:.1f} px)")

# Inisialisasi Tracker (EMA smoothing, aspect corection, unwrap angle)
tracker = PupilTemporalTracker(
    alpha=0.35, max_dev_ratio=0.08, history_len=5, 
    aspect_scale=calib_data.get("aspect_scale", 1.0)
)

last_center = None
last_diameter = None
no_det_streak = 0

# Penyiapan Debug Log CSV/TSV
debug_log_path = os.path.join(dir_mask, f"{vid_name}_debug_log.txt")
debug_headers = [
    "frame_index", "frame_name", "status", 
    "raw_center_x", "raw_center_y", "raw_axis_1", "raw_axis_2", "raw_angle", "raw_diameter",
    "smooth_center_x", "smooth_center_y", "smooth_axis_1", "smooth_axis_2", "smooth_angle", "smooth_diameter",
    "threshold", "candidate_score", "contrast", "eye_opening_px", "eye_roi_mean", 
    "anchor_val", "chosen_offset", "rejection_reason"
]

debug_rows = []

print(f"[INFO] Memulai Segmentasi {len(gray_files)} frame (Simpan Mask, Overlay & Log ke Drive)...")

# --- Fase Pelacakan ---
for i, filename in enumerate(tqdm(gray_files, desc=f"Segmentasi [{vid_name}]")):
    img_path = os.path.join(dir_gray, filename)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        diameters.append(0)
        centers.append((0, 0))
        angles.append(0)
        ellipses.append(None)
        masks_in_memory.append(np.zeros((800, 800), dtype=np.uint8))
        gray_frames_in_memory.append(np.zeros((800, 800), dtype=np.uint8))
        debug_rows.append(f"{i}\t{filename}\tREAD_ERROR\t" + "\t".join([""]*20))
        continue
        
    # --- PROSES DETEKSI ---
    blur_img = cv2.GaussianBlur(img, (21, 21), 0)
    min_val, _, min_loc, _ = cv2.minMaxLoc(blur_img)
    anchor_val = min_val
    
    blink_metrics = measure_blink_metrics(img)
    is_blk = is_blink(blink_metrics, anchor_val, blink_mean_gate)
    
    success = False
    chosen_pick = None
    rejection_reason = ""
    
    if is_blk:
        rejection_reason = "BLINK"
        last_center = None
        last_diameter = None
        no_det_streak = 0
        tracker.reset()
    else:
        anchors = build_track_anchors(blur_img, last_center)
        expected = last_diameter if last_diameter is not None else baseline
        
        for a_val, a_xy in anchors:
            img_proc = preprocess_glint(img) if cfg["glint_removal"] else img.copy()
            rows = _sweep_rows(img_proc, a_val, a_xy)
            zone = _stable_zone(rows, a_val, cfg["dark_margin"], cfg["dia_floor"])
            
            if zone:
                valid_zone = []
                for z in zone:
                    if z["inside"] <= min_val + GLOBAL_INSIDE_MARGIN and z["dia"] >= cfg["dia_floor"] * 0.85:
                        valid_zone.append(z)
                zone = valid_zone
                
            if not zone:
                rejection_reason = rejection_reason or "NO_ZONE"
                continue
                
            pick = _zone_pick(zone, expected_dia=expected)
            if not (dia_min <= pick["dia"] <= dia_max):
                rejection_reason = rejection_reason or "DIA_GATE"
                continue
            if last_center is not None:
                shift = math.hypot(pick["center"][0] - last_center[0], pick["center"][1] - last_center[1])
                if shift > MAX_CENTER_SHIFT_TRACK:
                    rejection_reason = rejection_reason or "SHIFT_GATE"
                    continue
            min_contrast = max(4.0, 0.12 * calib_data.get("contrast_med", 20.0))
            if pick["contrast"] < min_contrast:
                rejection_reason = rejection_reason or "CONTRAST_GATE"
                continue
                
            success = True
            chosen_pick = pick
            break

    # --- PENCATATAN HASIL & DRAWING MASK + OVERLAY ---
    mask_frame = np.zeros_like(img)
    overlay_frame = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    status_str = "SUCCESS" if success else "NO_DETECTION"
    if is_blk: status_str = "BLINK"
    
    if success and chosen_pick is not None:
        last_center = chosen_pick["center"]
        last_diameter = chosen_pick["dia"]
        no_det_streak = 0
        
        smooth_res = tracker.update(
            chosen_pick["center"], chosen_pick["axes"], 
            chosen_pick["angle"], chosen_pick["dia"]
        )
        
        sm_center = smooth_res["center"]
        sm_axes = smooth_res["axes"]
        sm_angle = smooth_res["angle"]
        sm_dia = smooth_res["diameter"]
        sm_ellipse = smooth_res["ellipse"]
        
        diameters.append(sm_dia)
        centers.append((int(sm_center[0]), int(sm_center[1])))
        angles.append(round(sm_angle, 2))
        ellipses.append(sm_ellipse)
        
        cv2.ellipse(mask_frame, sm_ellipse, 255, thickness=cv2.FILLED)
        cv2.ellipse(overlay_frame, sm_ellipse, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.circle(overlay_frame, (int(sm_center[0]), int(sm_center[1])), 3, (0, 255, 0), -1, cv2.LINE_AA)
        
        # Logging
        row_str = f"{i}\t{filename}\t{status_str}\t{chosen_pick['center'][0]:.6f}\t{chosen_pick['center'][1]:.6f}\t{chosen_pick['axes'][0]:.6f}\t{chosen_pick['axes'][1]:.6f}\t{chosen_pick['angle']:.6f}\t{chosen_pick['dia']:.6f}\t{sm_center[0]:.6f}\t{sm_center[1]:.6f}\t{sm_axes[0]:.6f}\t{sm_axes[1]:.6f}\t{sm_angle:.6f}\t{sm_dia:.6f}\t{chosen_pick['threshold']:.6f}\t{chosen_pick.get('score', 0.0):.6f}\t{chosen_pick['contrast']:.6f}\t{blink_metrics['eye_opening_px']}\t{blink_metrics['eye_roi_mean']:.6f}\t{anchor_val}\t{chosen_pick['off']}\t{rejection_reason}"
        debug_rows.append(row_str)
    else:
        no_det_streak += 1
        if no_det_streak >= NO_DET_STREAK_RESET:
            last_center = None
            tracker.reset()
            
        diameters.append(0)
        centers.append((0, 0))
        angles.append(0)
        ellipses.append(None)
        
        cv2.putText(overlay_frame, status_str, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        row_str = f"{i}\t{filename}\t{status_str}\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t{blink_metrics['eye_opening_px']}\t{blink_metrics['eye_roi_mean']:.6f}\t{anchor_val}\t\t{rejection_reason}"
        debug_rows.append(row_str)

    masks_in_memory.append(mask_frame)
    gray_frames_in_memory.append(img)
    
    cv2.imwrite(os.path.join(sub_mask, filename), mask_frame)
    cv2.imwrite(os.path.join(sub_overlay, filename), overlay_frame)

# Tulis file log debug
with open(debug_log_path, "w") as f:
    f.write("\t".join(debug_headers) + "\n")
    f.write("\n".join(debug_rows) + "\n")

print(f"[INFO] Kalibrasi, Segmentasi, dan penulisan Masker Biner, Overlay + Log Debug selesai.")


In [ ]:
# @title 4️⃣3️⃣ SOP-04 (Langkah 3): Render Berkas Video Verifikasi MP4 (1 Video Terpadu Utama)
# ======================================================================
# SOP-04 (Langkah 3): RENDERING 1 VIDEO TERPADU UTAMA
# ======================================================================
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

print(f"\n[INFO] Memulai perenderan berkas 1 Video Verifikasi Terpadu MP4...")

gray_files = sorted(os.listdir(dir_gray))

# === OUTLIER REJECTION & TREND ===
raw_d = np.array(diameters, dtype=np.float64)
raw_d[raw_d == 0] = np.nan

temp = np.copy(raw_d)
mask_nan = np.isnan(temp)
if np.any(~mask_nan):
    temp[mask_nan] = np.interp(np.flatnonzero(mask_nan), np.flatnonzero(~mask_nan), temp[~mask_nan])
else:
    temp[:] = 92.0

fps_asumsi = 50.0
window = 150
pad_w = window // 2
padded = np.pad(temp, (pad_w, pad_w), mode='edge')
trend = np.zeros_like(temp)
for i in range(len(temp)):
    trend[i] = np.median(padded[i:i+window])
    
abs_diff = np.abs(raw_d - trend) / (trend + 1e-5)
outliers = (abs_diff > 0.15) | np.isnan(raw_d)

kernel = np.ones(11, dtype=bool)
dilated_outliers = np.convolve(outliers, kernel, mode='same') > 0

clean_d = np.copy(raw_d)
clean_d[dilated_outliers] = np.nan

s_diams = pd.Series(clean_d).interpolate(limit_direction='both')
if s_diams.isna().all(): s_diams = pd.Series([92.0]*len(clean_d))
s_diams = s_diams.bfill().ffill()

s_diams_smooth = s_diams.rolling(window=5, min_periods=1, center=True).mean()
s_trend_smooth = pd.Series(trend).rolling(window=15, min_periods=1, center=True).mean()

d_array = s_diams_smooth.to_numpy()
peaks, _ = find_peaks(d_array, distance=15, prominence=0.5)

baseline = np.mean(d_array)
max_d = np.max(d_array)
min_d = np.min(d_array)
amp_pct = ((max_d - min_d) / baseline) * 100 if baseline > 0 else 0
freq = (len(peaks) / len(d_array)) * fps_asumsi

# === RENDERING 1 VIDEO TERPADU (Sama seperti lokal) ===
path_v1 = os.path.join(dir_mask, f"{vid_name}_Video_Tracking_dan_Grafik_Realtime.mp4")

FRAME_W, FRAME_H = 800, 800
GRAPH_W, GRAPH_H = 1600, 600
CANVAS_W, CANVAS_H = 1600, 1400
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer_v1 = cv2.VideoWriter(path_v1, fourcc, fps_asumsi, (CANVAS_W, CANVAS_H))

g_max = max(np.max(d_array), np.max(trend)) + 5
g_min = min(np.min(d_array), np.min(trend)) - 5

for i in tqdm(range(len(gray_files)), desc="Rendering Video Terpadu Utama"):
    roi_gray = gray_frames_in_memory[i]
    mask_frame = masks_in_memory[i]

    bgr_roi = cv2.cvtColor(roi_gray, cv2.COLOR_GRAY2BGR)
    if bgr_roi.shape[0] != FRAME_H or bgr_roi.shape[1] != FRAME_W:
        bgr_roi = cv2.resize(bgr_roi, (FRAME_W, FRAME_H))

    raw_cx, raw_cy = centers[i]
    raw_diam = diameters[i]
    is_blink = dilated_outliers[i]

    if is_blink or raw_diam == 0:
        cv2.putText(bgr_roi, "BLINK", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
    else:
        overlay_radius = int(raw_diam / 2.0)
        cv2.circle(bgr_roi, (raw_cx, raw_cy), overlay_radius, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.circle(bgr_roi, (raw_cx, raw_cy), 3, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(bgr_roi, f"PUPIL D={raw_diam:.1f}px", (max(10, raw_cx-80), max(30, raw_cy-overlay_radius-15)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
    
    cv2.putText(bgr_roi, f"1. Original ROI + Tracking [Frame {i}]", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    mask_bgr = cv2.cvtColor(mask_frame, cv2.COLOR_GRAY2BGR)
    if mask_bgr.shape[0] != FRAME_H or mask_bgr.shape[1] != FRAME_W:
        mask_bgr = cv2.resize(mask_bgr, (FRAME_W, FRAME_H))
    cv2.putText(mask_bgr, "2. Realtime Binary Mask", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    top_row = np.hstack((bgr_roi, mask_bgr))

    fig, ax = plt.subplots(figsize=(16, 6), dpi=100)
    x_vals = np.arange(i + 1)
    ax.plot(x_vals, d_array[:i+1], color='#3176b5', linewidth=2.5, label='Diameter Pupil (px)')
    ax.plot(x_vals, s_trend_smooth.iloc[:i+1], color='#f13c3c', linestyle='--', linewidth=2, label='Trend Baseline')
    
    peaks_up_to_i = [p for p in peaks if p <= i]
    if peaks_up_to_i:
        ax.scatter(peaks_up_to_i, d_array[peaks_up_to_i], color='#e87a20', s=70, zorder=5, label='Puncak Hippus')
        
    ax.set_xlim(0, len(gray_files))
    ax.set_ylim(g_min, g_max)
    ax.set_title(f"Temporal Analysis (Pupillary Hippus) - Responden {vid_name} | Frekuensi: {freq:.2f} Hz | Fluktuasi: {amp_pct:.2f} %", fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel("Frame ID (50 FPS)", fontsize=11)
    ax.set_ylabel("Diameter Pupil (px)", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=10)
    
    fig.tight_layout()
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    graph_img = cv2.cvtColor(rgba, cv2.COLOR_RGBA2BGR)
    plt.close(fig)

    if graph_img.shape[1] != GRAPH_W or graph_img.shape[0] != GRAPH_H:
        graph_img = cv2.resize(graph_img, (GRAPH_W, GRAPH_H))

    canvas = np.vstack((top_row, graph_img))
    writer_v1.write(canvas)

writer_v1.release()

print("======================================================================")
print(" RENDER 1 VIDEO TERPADU UTAMA SELESAI!")
print("Folder Utama Drive : 3_Mask_Pupil_SOP04")
print("======================================================================")


In [ ]:
# @title 5️⃣ SOP-05: Tracking Pupil & Analisis Sinyal Temporal Fluktuasi Hippus
# ======================================================================
# SOP-05: ANALISIS TEMPORAL HIPPUS & EKSPOR DATA (SINKRON DENGAN SOP-04)
# ======================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-05: ANALISIS TEMPORAL HIPPUS & EKSPOR DATA")
print(f"======================================================================")

cx_list = [c[0] for c in centers]
cy_list = [c[1] for c in centers]
df = pd.DataFrame({
    'Frame': range(len(diameters)),
    'Center_X': cx_list,
    'Center_Y': cy_list,
    'Ellipse_Angle': angles,
    'Diameter_px': diameters
})

# 1. Filter Outlier & Interpolasi (Logika Presisi Identik SOP-04)
fps_asumsi = 50.0 # Default fallback, 50.0 atau 90.0 bisa digunakan
raw_d = df['Diameter_px'].replace(0, np.nan).to_numpy(dtype=np.float64)

temp = np.copy(raw_d)
mask_nan = np.isnan(temp)
if np.any(~mask_nan):
    temp[mask_nan] = np.interp(np.flatnonzero(mask_nan), np.flatnonzero(~mask_nan), temp[~mask_nan])
else:
    temp[:] = 92.0

window = 150
pad_w = window // 2
padded = np.pad(temp, (pad_w, pad_w), mode='edge')
trend = np.zeros_like(temp)
for i in range(len(temp)):
    trend[i] = np.median(padded[i:i+window])
    
abs_diff = np.abs(raw_d - trend) / (trend + 1e-5)
outliers = (abs_diff > 0.15) | np.isnan(raw_d)

kernel = np.ones(11, dtype=bool)
dilated_outliers = np.convolve(outliers, kernel, mode='same') > 0

df['Is_Blink'] = dilated_outliers.astype(int)

clean_d = np.copy(raw_d)
clean_d[dilated_outliers] = np.nan

s_diams = pd.Series(clean_d).interpolate(limit_direction='both')
if s_diams.isna().all(): s_diams = pd.Series([92.0]*len(clean_d))
s_diams = s_diams.bfill().ffill()

# Penapisan Mulus Sesuai SOP-04 (window=5, window=15)
s_diams_smooth = s_diams.rolling(window=5, min_periods=1, center=True).mean()
s_trend_smooth = pd.Series(trend).rolling(window=15, min_periods=1, center=True).mean()

df['Filtered_Diameter_px'] = s_diams_smooth.values
df['Trend_Baseline'] = s_trend_smooth.values

# 2. Ekstraksi Fitur Medis Hippus Identik SOP-04 (distance=15, prominence=0.5)
d_array = s_diams_smooth.to_numpy()
peaks, _ = find_peaks(d_array, distance=15, prominence=0.5)

baseline = np.mean(d_array)
max_d = np.max(d_array)
min_d = np.min(d_array)
amplitude = ((max_d - min_d) / baseline) * 100 if baseline > 0 else 0.0
duration_sec = len(df) / fps_asumsi
frequency = float(len(peaks) / duration_sec) if duration_sec > 0 else 0.0

print(f"\n=== HASIL ANALISIS HIPPUS [{vid_name}] ===")
print(f"- Frekuensi Hippus : {frequency:.2f} Hz (Gelombang/detik)")
print(f"- Fluktuasi Amplitudo: {amplitude:.2f} % (Rentang fluktuasi pupil)")
print(f"- Rata-rata Diameter: {baseline:.2f} px\n")

dir_hasil = os.path.join(ROOT_DIR, f"{vid_name}/4_Hasil_Analisis_SOP05_06")
os.makedirs(dir_hasil, exist_ok=True)

csv_path = os.path.join(dir_hasil, f"{vid_name}_laporan_analisis_sop06.csv")
df.to_csv(csv_path, index=False)

# Render Grafik PNG Identik dengan Visual SOP-04 Video
plt.figure(figsize=(12, 5), dpi=120)
x_vals = df['Frame']
plt.plot(x_vals, df['Filtered_Diameter_px'], color='#3176b5', linewidth=2, label='Diameter Pupil (px)')
plt.plot(x_vals, df['Trend_Baseline'], color='#f13c3c', linestyle='--', linewidth=2, label='Trend Baseline')

if len(peaks) > 0:
    plt.scatter(x_vals.iloc[peaks], df['Filtered_Diameter_px'].iloc[peaks], color='#e87a20', s=60, zorder=5, label='Puncak Hippus')

g_max = max(np.max(d_array), np.max(trend)) + 5
g_min = max(0, min(np.min(d_array), np.min(trend)) - 5)
plt.ylim(g_min, g_max)

plt.title(f'Temporal Analysis (Pupillary Hippus) - Responden {vid_name}\nFrekuensi: {frequency:.2f} Hz | Fluktuasi: {amplitude:.2f} %', fontsize=12, fontweight='bold')
plt.xlabel('Frame Number (50 FPS)')
plt.ylabel('Diameter (px)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()

graph_path = os.path.join(dir_hasil, f"{vid_name}_chart_hippus_sop06.png")
plt.savefig(graph_path)
plt.show()

print(f"\n[INFO] Data CSV dan Grafik Hippus Progresif berhasil disimpan ke Google Drive.")


In [ ]:
# @title 6️⃣ SOP-06: Generator Laporan PDF Diagnostik Klinis Individual
# ======================================================================
# SOP-06: GENERATOR LAPORAN PDF DIAGNOSTIK KLINIS (REPORTLAB FAILSAFE)
# ======================================================================
import os
import pandas as pd
import numpy as np

print(f"======================================================================")
print(f"[RESPONDEN: {vid_name}] - SOP-06: GENERATOR LAPORAN PDF DIAGNOSTIK KLINIS")
print(f"======================================================================")

# 1. Pastikan Pustaka ReportLab Tersedia (Failsafe Anti-Crash)
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
except ImportError:
    import subprocess
    print("[INFO] Menginstal pemustakaan reportlab...")
    subprocess.check_call(['pip', 'install', 'reportlab', '--quiet'])
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors

dir_hasil = os.path.join(ROOT_DIR, f"{vid_name}/4_Hasil_Analisis_SOP05_06")
os.makedirs(dir_hasil, exist_ok=True)

pdf_path = os.path.join(dir_hasil, f"{vid_name}_Laporan_Klinis_SOP06.pdf")
graph_path = os.path.join(dir_hasil, f"{vid_name}_chart_hippus_sop06.png")

# 2. Persiapan Data Metrik dari SOP-05
blink_count = int(df['Is_Blink'].sum()) if 'Is_Blink' in df.columns else 0
total_f = len(df)
blink_rate_pct = (blink_count / total_f * 100.0) if total_f > 0 else 0.0

doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()

title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=16, leading=20, textColor=colors.HexColor('#003366'))
sub_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=9.5, leading=13, textColor=colors.HexColor('#444444'))
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontName='Helvetica', fontSize=9, leading=12, textColor=colors.HexColor('#222222'))

story = []
story.append(Paragraph(f"LAPORAN DIAGNOSTIK KLINIS INDIVIDUAL - RESPONDEN {vid_name}", title_style))
story.append(Paragraph(f"<b>Platform:</b> Pupil Dataset Processor (PDP) | <b>Hibah Penelitian:</b> 2026", sub_style))
story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#003366"), spaceAfter=10))

tbl_data = [
    [Paragraph("<b>Parameter Metrik Medis</b>", body_style), Paragraph("<b>Nilai Hasil Analisis</b>", body_style)],
    [Paragraph("Identitas Responden", body_style), Paragraph(f"{vid_name}", body_style)],
    [Paragraph("Total Frame Teranalisis", body_style), Paragraph(f"{total_f} Frame (50 FPS)", body_style)],
    [Paragraph("Rata-rata Diameter Baseline", body_style), Paragraph(f"{baseline:.2f} Piksel (px)", body_style)],
    [Paragraph("Frekuensi Osilasi Hippus", body_style), Paragraph(f"{frequency:.2f} Hz (Gelombang/detik)", body_style)],
    [Paragraph("Fluktuasi Amplitudo Pupil", body_style), Paragraph(f"{amplitude:.2f} %", body_style)],
    [Paragraph("Persentase Kedipan (Blink Rate)", body_style), Paragraph(f"{blink_rate_pct:.2f} % ({blink_count} Frame)", body_style)]
]

t = Table(tbl_data, colWidths=[280, 260])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#EAECEE")),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#BDC3C7")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE')
]))
story.append(t)
story.append(Spacer(1, 15))

if os.path.exists(graph_path):
    story.append(Paragraph("Grafik Deret-Waktu Sinyal Fluktuasi Pupil (Hippus):", styles['Heading2']))
    story.append(Spacer(1, 5))
    story.append(RLImage(graph_path, width=540, height=225))

doc.build(story)

print(f"======================================================================")
print(f"📄 Nama Berkas PDF: {os.path.basename(pdf_path)}")
print(f"📍 Drive Local Path: {pdf_path}")
print(f"🔗 Akses Google Drive: https://drive.google.com/drive/my-drive")
print(f"======================================================================")


## <font color="#38b6ff" face="Palatino Linotype">**FASE 3: VALIDASI BENCHMARK LPW & PENYELESAIAN RISET**</font>

<details>
<summary>📌 <b>Lihat Infografis FASE 3: Validasi Benchmark & Evaluasi (Klik untuk Membuka/Menutup)</b></summary>
<br>
<p align="center">
  <img src="https://github.com/cilacap-loki/pupil-dataset-analysis/raw/main/assets/img/fase3_visualisasi_garis_besar.png?raw=true" width="100%" alt="FASE 3 Infographic">
</p>
</details>

Tahap pengujian mutlak (*cross-validation*) untuk mengukur seberapa akurat algoritma OpenCV kita jika disandingkan dengan label *Ground Truth* publik berstandar internasional (*Labeled Pupils in the Wild* / LPW), serta tahap finalisasi tata kelola dan dokumentasi penelitian.

* **SOP-07 (Validasi Benchmark LPW):** Memutar *dataset* sekunder 90 FPS yang sarat *noise* & gerakan agresif, membandingkannya dengan koordinat *Ground Truth* manual, merender visualisasi komparatif, serta mencetak laporan kuantitatif *Detection Rate* secara absolut.
* **SOP-08 (Penyimpanan & Tata Kelola Dataset):** Pengarsipan seluruh hasil luaran (video, CSV, PDF) ke dalam basis data riset (*Google Drive / Harddisk*) yang berstatus *Restricted / Private*.
* **SOP-09 (Dokumentasi Riset & Publikasi Jurnal):** Penggunaan data luaran SOP 1 hingga 8 sebagai basis data empiris untuk menyusun naskah manuskrip jurnal ilmiah dan laporan akhir riset.


In [ ]:
# @title 7️⃣1️⃣ SOP-07 (Langkah 1): Form Pemilihan Dataset LPW
# ======================================================================
# SOP-07 (Langkah 1): PEMILIHAN DATASET LPW (DRIVE AUTOMATIC SCAN)
# ======================================================================
import os
import glob
import ipywidgets as widgets
from IPython.display import display

print("======================================================================")
print(" SOP-07: LPW BENCHMARK VALIDATION (GROUND TRUTH EVALUATION)")
print("======================================================================")

# 1. Deteksi Presisi Jalur LPW Dataset di Google Drive
candidate_paths = [
    "/content/drive/MyDrive/Hibah Penelitian/LPW Dataset",
    os.path.join(HIBAH_ROOT if 'HIBAH_ROOT' in globals() else "/content/drive/MyDrive/Hibah Penelitian", "LPW Dataset"),
    os.path.join(ROOT_DIR if 'ROOT_DIR' in globals() else "/content/drive/MyDrive/Hibah Penelitian/Outputs", "Benchmark_LPW"),
    os.path.join(ROOT_DIR if 'ROOT_DIR' in globals() else "/content/drive/MyDrive/Hibah Penelitian/Outputs", "LPW Dataset")
]

LPW_ROOT = None
for path in candidate_paths:
    if os.path.exists(path):
        subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)) and not d.startswith("Output")]
        if subdirs:
            LPW_ROOT = path
            break

if not LPW_ROOT:
    LPW_ROOT = "/content/drive/MyDrive/Hibah Penelitian/LPW Dataset"
    os.makedirs(LPW_ROOT, exist_ok=True)
    print(f"[WARN] Folder LPW Dataset disiapkan di: {LPW_ROOT}")
else:
    print(f"[INFO] Berhasil mendeteksi LPW Dataset di: {LPW_ROOT}")

# 2. Deteksi Subfolder Responden LPW (001, 004, 009)
try:
    available_lpw = [d for d in os.listdir(LPW_ROOT) if os.path.isdir(os.path.join(LPW_ROOT, d)) and not d.startswith("Output")]
    available_lpw = sorted(available_lpw)
except FileNotFoundError:
    available_lpw = []

if not available_lpw:
    available_lpw = ["001", "004", "009"]

dropdown_lpw = widgets.Dropdown(
    options=available_lpw,
    value=available_lpw[0],
    description='Target LPW:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

display(dropdown_lpw)
print("\n[INFO] Pilih Responden LPW dari dropdown di atas, lalu jalankan Langkah 2.")


In [ ]:
# @title 🎥 SOP-07 (Langkah 2): Parsing Dataset & Render Video Komparasi 4-Panel
# ======================================================================
# SOP-07 (Langkah 2): PARSING GROUND TRUTH LPW DENGAN NAMA FILE DINAMIS
# ======================================================================
import os
import pandas as pd
import numpy as np
import cv2, math
import matplotlib.pyplot as plt
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

lpw_id = dropdown_lpw.value
dir_lpw = os.path.join(LPW_ROOT, str(lpw_id))

file_prefix = str(int(lpw_id)) 
gt_file = os.path.join(dir_lpw, f"{file_prefix}.txt")
vid_lpw_path = os.path.join(dir_lpw, f"{file_prefix}.avi")

print(f"======================================================================")
print(f"[TARGET LPW: {lpw_id}] - MEMUAT GROUND TRUTH & VIDEO")
print(f"======================================================================")

if os.path.exists(gt_file):
    try:
        gt_df = pd.read_csv(gt_file, sep=r'\s+', header=None, names=['GT_X', 'GT_Y'])
        print(f"[INFO] Berhasil memuat {len(gt_df)} frame dari Ground Truth ({file_prefix}.txt).")
    except Exception as e:
        print(f"[ERROR] Gagal mem-parsing file GT: {e}")
        gt_df = pd.DataFrame(columns=['GT_X', 'GT_Y'])
else:
    print(f"[ERROR] Berkas Ground Truth ({file_prefix}.txt) tidak ditemukan di {dir_lpw}")
    gt_df = pd.DataFrame(columns=['GT_X', 'GT_Y'])

if os.path.exists(vid_lpw_path):
    print(f"[INFO] Video LPW ditemukan: {file_prefix}.avi")
else:
    print(f"[ERROR] Video {file_prefix}.avi tidak ditemukan di {dir_lpw}")

out_dir_lpw = os.path.join(dir_lpw, f"Output_LPW_{lpw_id}")
os.makedirs(out_dir_lpw, exist_ok=True)
print(f"[INFO] Folder output siap di: {out_dir_lpw}")

# ======================================================================
#  BAGIAN 2: RENDERING VIDEO MP4 GRID 4-PANEL DENGAN V3 ENGINE
# ======================================================================

print(f"\n[INFO] Mengeksekusi Analisis V3 dan Merender Video MP4 Grid 4-Panel (2x2)...")

if 'run_calibration' not in globals():
    raise NameError("[ERROR] Fungsi V3 belum tersedia. Jalankan SOP-04 (Langkah 1) terlebih dahulu!")

cap = cv2.VideoCapture(vid_lpw_path)
if not cap.isOpened():
    raise ValueError(f"Tidak dapat membuka video {vid_lpw_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps): fps = 95.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ret, first_frame = cap.read()
if not ret:
    raise ValueError("Video kosong.")
H_vid, W_vid, _ = first_frame.shape

PANEL_W, PANEL_H = W_vid, H_vid
GRID_W, GRID_H = PANEL_W * 2, PANEL_H * 2

valid_gt_x = gt_df['GT_X'].dropna()
valid_gt_y = gt_df['GT_Y'].dropna()
valid_gt_x = valid_gt_x[valid_gt_x > 0]
valid_gt_y = valid_gt_y[valid_gt_y > 0]

if len(valid_gt_x) > 0:
    x_min_lim = max(0, float(valid_gt_x.min()) - 25)
    x_max_lim = min(float(W_vid), float(valid_gt_x.max()) + 25)
else:
    x_min_lim, x_max_lim = 0, float(W_vid)

if len(valid_gt_y) > 0:
    y_min_lim = max(0, float(valid_gt_y.min()) - 25)
    y_max_lim = min(float(H_vid), float(valid_gt_y.max()) + 25)
else:
    y_min_lim, y_max_lim = 0, float(H_vid)

print(f"[INFO] Batas Sumbu X Tetap : [{x_min_lim:.1f} px s/d {x_max_lim:.1f} px]")
print(f"[INFO] Batas Sumbu Y Tetap : [{y_min_lim:.1f} px s/d {y_max_lim:.1f} px]")

print(f"[INFO] Menyiapkan Kalibrasi V3 untuk LPW {lpw_id}...")
tmp_calib_dir = os.path.join(out_dir_lpw, "tmp_calib")
os.makedirs(tmp_calib_dir, exist_ok=True)

dummy_files = [f"frame_{i}.png" for i in range(total_frames)]
CALIB_N = 60
indices = [int(i * total_frames / CALIB_N) for i in range(CALIB_N)]
indices = sorted(set(min(i, total_frames - 1) for i in indices))

for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(os.path.join(tmp_calib_dir, dummy_files[idx]), cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))

calib_data = run_calibration(tmp_calib_dir, dummy_files)
cfg = STRATEGIES[calib_data["strategy"]]
dia_min = calib_data["dia_min"]
dia_max = calib_data["dia_max"]
blink_mean_gate = calib_data["blink_mean_gate"]
baseline = calib_data["baseline_dia"]
conf = calib_data["confidence"]
print(f"   => Strategi Terpilih: {calib_data['strategy'].upper()} (Conf: {conf:.3f}, Baseline: {baseline:.1f} px)")

tracker = PupilTemporalTracker(
    alpha=0.35, max_dev_ratio=0.08, history_len=5, 
    aspect_scale=calib_data.get("aspect_scale", 1.0)
)
last_center = None
last_diameter = None
no_det_streak = 0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Super_Komparasi.mp4")
out_video = cv2.VideoWriter(out_vid_path, fourcc, fps, (GRID_W, GRID_H))

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

pred_x, pred_y, pred_d_list = [], [], []
euclidean_errors = []
limit_frames = min(total_frames, len(gt_df)) if len(gt_df) > 0 else total_frames

for i in tqdm(range(limit_frames), desc=f"Evaluasi LPW {lpw_id}"):
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    blur_img = cv2.GaussianBlur(gray, (21, 21), 0)
    min_val, _, min_loc, _ = cv2.minMaxLoc(blur_img)
    anchor_val = min_val
    
    blink_metrics = measure_blink_metrics(gray)
    is_blk = is_blink(blink_metrics, anchor_val, blink_mean_gate)
    
    success = False
    chosen_pick = None
    
    if is_blk:
        last_center = None
        last_diameter = None
        no_det_streak = 0
        tracker.reset()
    else:
        anchors = build_track_anchors(blur_img, last_center)
        expected = last_diameter if last_diameter is not None else baseline
        
        for a_val, a_xy in anchors:
            img_proc = preprocess_glint(gray) if cfg["glint_removal"] else gray.copy()
            rows = _sweep_rows(img_proc, a_val, a_xy)
            zone = _stable_zone(rows, a_val, cfg["dark_margin"], cfg["dia_floor"])
            
            if zone:
                valid_zone = [z for z in zone if z["inside"] <= min_val + GLOBAL_INSIDE_MARGIN and z["dia"] >= cfg["dia_floor"] * 0.85]
                zone = valid_zone
                
            if not zone: continue
            
            pick = _zone_pick(zone, expected_dia=expected)
            if not (dia_min <= pick["dia"] <= dia_max): continue
            
            if last_center is not None:
                shift = math.hypot(pick["center"][0] - last_center[0], pick["center"][1] - last_center[1])
                if shift > MAX_CENTER_SHIFT_TRACK: continue
                
            min_contrast = max(4.0, 0.12 * calib_data.get("contrast_med", 20.0))
            if pick["contrast"] < min_contrast: continue
                
            success = True
            chosen_pick = pick
            break

    cx, cy, diam, best_ellipse = 0, 0, 0, None
    if success and chosen_pick is not None:
        last_center = chosen_pick["center"]
        last_diameter = chosen_pick["dia"]
        no_det_streak = 0
        
        smooth_res = tracker.update(
            chosen_pick["center"], chosen_pick["axes"], 
            chosen_pick["angle"], chosen_pick["dia"]
        )
        cx, cy = smooth_res["center"]
        diam = smooth_res["diameter"]
        best_ellipse = smooth_res["ellipse"]
    else:
        no_det_streak += 1
        if no_det_streak >= NO_DET_STREAK_RESET:
            last_center = None
            tracker.reset()
            
    pred_x.append(cx)
    pred_y.append(cy)
    pred_d_list.append(diam)
    
    if i < len(gt_df):
        gx = gt_df['GT_X'].iloc[i]
        gy = gt_df['GT_Y'].iloc[i]
    else:
        gx, gy = np.nan, np.nan
        
    if cx > 0 and not np.isnan(gx) and not np.isnan(gy) and gx > 0:
        err = np.hypot(cx - gx, cy - gy)
    else:
        err = np.nan
    euclidean_errors.append(err)
    
    tl_frame = frame.copy()
    if not np.isnan(gx) and not np.isnan(gy) and gx > 0:
        cv2.circle(tl_frame, (int(gx), int(gy)), 4, (0, 255, 0), -1, cv2.LINE_AA)
        cv2.drawMarker(tl_frame, (int(gx), int(gy)), (0, 0, 255), cv2.MARKER_CROSS, 14, 2)
        cv2.putText(tl_frame, f"1. GT LPW CENTER (X:{int(gx)}, Y:{int(gy)})", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
    else:
        cv2.putText(tl_frame, "1. GT LPW CENTER (BLINK / NO DATA)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)
        
    bl_frame = frame.copy()
    if cx > 0:
        if best_ellipse is not None and len(best_ellipse) == 3:
            cv2.ellipse(bl_frame, best_ellipse, (255, 0, 0), 2, cv2.LINE_AA)
        else:
            cv2.circle(bl_frame, (int(cx), int(cy)), int(diam/2), (255, 0, 0), 2, cv2.LINE_AA)
        cv2.circle(bl_frame, (int(cx), int(cy)), 3, (0, 0, 255), -1, cv2.LINE_AA)
        cv2.putText(bl_frame, f"2. V3 TRACKING (D:{diam:.1f}px)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
        if not np.isnan(err):
            cv2.putText(bl_frame, f"Err:{err:.1f}px", (PANEL_W - 150, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2, cv2.LINE_AA)
    else:
        cv2.putText(bl_frame, "2. V3 TRACKING (BLINK / LOST)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)

    fig_x, ax_x = plt.subplots(figsize=(PANEL_W/100, PANEL_H/100), dpi=100)
    x_indices = np.arange(i + 1)
    gt_x_vals = gt_df['GT_X'].iloc[:i+1].values if len(gt_df) >= i+1 else np.full(i+1, np.nan)
    pred_x_vals = np.array([x if x > 0 else np.nan for x in pred_x[:i+1]])
    
    ax_x.plot(x_indices, gt_x_vals, color='#38a169', linewidth=2, label='Ground Truth X (1.txt)')
    ax_x.plot(x_indices, pred_x_vals, color='#e53e3e', linestyle='--', linewidth=2, label='V3 Prediksi X')
    ax_x.set_xlim(0, limit_frames)
    ax_x.set_ylim(x_min_lim, x_max_lim)
    ax_x.set_title(f"3. Perbandingan Posisi Sumbu X - Frame {i:04d}", fontsize=11, fontweight='bold', pad=8)
    ax_x.set_xlabel("Frame ID", fontsize=9)
    ax_x.set_ylabel("Koordinat X (px)", fontsize=9)
    ax_x.grid(True, linestyle='--', alpha=0.5)
    ax_x.legend(loc='upper right', fontsize=8)
    fig_x.tight_layout()
    
    fig_x.canvas.draw()
    tr_img = np.asarray(fig_x.canvas.buffer_rgba())[:, :, :3]
    tr_img = cv2.cvtColor(tr_img, cv2.COLOR_RGB2BGR)
    plt.close(fig_x)
    if tr_img.shape[0] != PANEL_H or tr_img.shape[1] != PANEL_W:
        tr_img = cv2.resize(tr_img, (PANEL_W, PANEL_H))

    fig_y, ax_y = plt.subplots(figsize=(PANEL_W/100, PANEL_H/100), dpi=100)
    gt_y_vals = gt_df['GT_Y'].iloc[:i+1].values if len(gt_df) >= i+1 else np.full(i+1, np.nan)
    pred_y_vals = np.array([y if y > 0 else np.nan for y in pred_y[:i+1]])
    
    ax_y.plot(x_indices, gt_y_vals, color='#38a169', linewidth=2, label='Ground Truth Y (1.txt)')
    ax_y.plot(x_indices, pred_y_vals, color='#3182ce', linestyle='--', linewidth=2, label='V3 Prediksi Y')
    ax_y.set_xlim(0, limit_frames)
    ax_y.set_ylim(y_min_lim, y_max_lim)
    ax_y.set_title(f"4. Perbandingan Posisi Sumbu Y - Frame {i:04d}", fontsize=11, fontweight='bold', pad=8)
    ax_y.set_xlabel("Frame ID", fontsize=9)
    ax_y.set_ylabel("Koordinat Y (px)", fontsize=9)
    ax_y.grid(True, linestyle='--', alpha=0.5)
    ax_y.legend(loc='upper right', fontsize=8)
    fig_y.tight_layout()
    
    fig_y.canvas.draw()
    br_img = np.asarray(fig_y.canvas.buffer_rgba())[:, :, :3]
    br_img = cv2.cvtColor(br_img, cv2.COLOR_RGB2BGR)
    plt.close(fig_y)
    if br_img.shape[0] != PANEL_H or br_img.shape[1] != PANEL_W:
        br_img = cv2.resize(br_img, (PANEL_W, PANEL_H))

    top_row = np.hstack([tl_frame, tr_img])
    bot_row = np.hstack([bl_frame, br_img])
    grid_canvas = np.vstack([top_row, bot_row])
    
    out_video.write(grid_canvas)

cap.release()
out_video.release()
print(f"\n[INFO] Render Video Grid 4-Panel (2x2) Selesai! Tersimpan di: {out_vid_path}")

In [ ]:
# @title 7️⃣3️⃣ SOP-07 (Langkah 3): Ekspor pred.txt, 3 Grafik PNG, CSV & PDF Report
# ======================================================================
# SOP-07 (Langkah 3): EKSPOR PRED.TXT, 3 GRAFIK PNG, CSV & REPORTLAB PDF
# ======================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"\n======================================================================")
print(f"[TARGET LPW: {lpw_id}] - EKSPOR PRED.TXT, 3 GRAFIK PNG, CSV & PDF REPORT")
print(f"======================================================================")

# 1. Ekspor Berkas pred.txt Murni 2-Kolom (Format standar LPW)
pred_txt_path = os.path.join(out_dir_lpw, "pred.txt")
with open(pred_txt_path, "w", encoding="utf-8") as f_pred:
    for x_val, y_val in zip(pred_x, pred_y):
        f_pred.write(f"{float(x_val):.2f} {float(y_val):.2f}\n")
print(f"✅ [1/5] Berkas pred.txt murni 2-kolom tersimpan di: {pred_txt_path}")

# 2. Simpan Spreadsheet CSV Metrics
df_metrics = pd.DataFrame({
    'Frame': range(1, len(euclidean_errors) + 1),
    'GT_X': gt_df['GT_X'].iloc[:len(euclidean_errors)].values if len(gt_df) >= len(euclidean_errors) else np.nan,
    'GT_Y': gt_df['GT_Y'].iloc[:len(euclidean_errors)].values if len(gt_df) >= len(euclidean_errors) else np.nan,
    'Pred_X': pred_x,
    'Pred_Y': pred_y,
    'Pred_D': pred_d_list,
    'Center_Error_px': euclidean_errors,
    'Is_Blink': [1 if d == 0 else 0 for d in pred_d_list]
})

csv_out_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Benchmark_Report.csv")
df_metrics.to_csv(csv_out_path, index=False)
print(f"✅ [2/5] Berkas Spreadsheet CSV tersimpan di: {csv_out_path}")

# 3. Hitung Metrik Kuantitatif
valid_errs = df_metrics['Center_Error_px'].dropna()
mean_err = float(valid_errs.mean()) if len(valid_errs) > 0 else 0.0
median_err = float(valid_errs.median()) if len(valid_errs) > 0 else 0.0
detection_rate_10px = float((valid_errs < 10.0).mean() * 100.0) if len(valid_errs) > 0 else 0.0

print(f"\n=== HASIL EVALUASI BENCHMARK LPW [{lpw_id}] ===")
print(f"- Mean Center Error (px)  : {mean_err:.2f} px")
print(f"- Median Center Error (px) : {median_err:.2f} px")
print(f"- Detection Rate (< 10px) : {detection_rate_10px:.2f} %")

# 4. Render 3 Berkas Grafik PNG Statis
# --- Grafik 1: Perbandingan Sumbu X ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['GT_X'], color='#38a169', linewidth=1.5, label='Ground Truth X (1.txt)')
plt.plot(df_metrics['Frame'], df_metrics['Pred_X'].replace(0, np.nan), color='#e53e3e', linestyle='--', linewidth=1.5, label='OpenCV Prediksi X')
plt.title(f'Perbandingan Sinyal Trajektori Sumbu X - LPW {lpw_id}', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Koordinat X (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_x_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Sumbu_X_Chart.png")
plt.savefig(chart_x_path, dpi=150)
plt.show()

# --- Grafik 2: Perbandingan Sumbu Y ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['GT_Y'], color='#38a169', linewidth=1.5, label='Ground Truth Y (1.txt)')
plt.plot(df_metrics['Frame'], df_metrics['Pred_Y'].replace(0, np.nan), color='#3182ce', linestyle='--', linewidth=1.5, label='OpenCV Prediksi Y')
plt.title(f'Perbandingan Sinyal Trajektori Sumbu Y - LPW {lpw_id}', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Koordinat Y (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_y_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Sumbu_Y_Chart.png")
plt.savefig(chart_y_path, dpi=150)
plt.show()

# --- Grafik 3: Ground Truth Distance Error ---
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(df_metrics['Frame'], df_metrics['Center_Error_px'], label='Center Distance Error (px)', color='#d53f8c', linewidth=1.2)
plt.axhline(10.0, color='#38a169', linestyle='--', linewidth=1.5, label='Threshold Toleransi 10px')
plt.title(f'Ground Truth Benchmark Validation LPW {lpw_id}\nMean Error: {mean_err:.2f} px | Detection Rate (<10px): {detection_rate_10px:.2f} %', fontsize=10, fontweight='bold')
plt.xlabel('Frame ID', fontsize=8)
plt.ylabel('Center Distance Error (px)', fontsize=8)
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
chart_error_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Error_Chart.png")
plt.savefig(chart_error_path, dpi=150)
plt.show()
print(f"✅ [3/5] Berhasil menyimpan 3 grafik PNG statis (Sumbu X, Sumbu Y, & Error Chart)!")

# 5. Menerbitkan Dokumen PDF Evaluasi Resmi (ReportLab)
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'reportlab', '--quiet'])
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, HRFlowable
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors

pdf_out_path = os.path.join(out_dir_lpw, f"LPW_{lpw_id}_Laporan_Evaluasi_GT.pdf")
doc = SimpleDocTemplate(pdf_out_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()

title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=16, leading=20, textColor=colors.HexColor('#003366'))
sub_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=9.5, leading=13, textColor=colors.HexColor('#444444'))
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontName='Helvetica', fontSize=9, leading=12, textColor=colors.HexColor('#222222'))

story = []
story.append(Paragraph(f"LAPORAN EVALUASI BENCHMARK LPW - DATASET {lpw_id}", title_style))
story.append(Paragraph(f"<b>Algoritma:</b> OpenCV Pupil Detector | <b>Dataset:</b> Labeled Pupils in the Wild (LPW {lpw_id})", sub_style))
story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#003366"), spaceAfter=8))

tbl_data = [
    [Paragraph("<b>Metrik Evaluasi</b>", body_style), Paragraph("<b>Nilai Hasil</b>", body_style)],
    [Paragraph("Mean Center Error (Rata-rata)", body_style), Paragraph(f"{mean_err:.2f} Piksel", body_style)],
    [Paragraph("Median Center Error (Nilai Tengah)", body_style), Paragraph(f"{median_err:.2f} Piksel", body_style)],
    [Paragraph("Detection Rate (Error < 10px)", body_style), Paragraph(f"{detection_rate_10px:.2f} %", body_style)],
    [Paragraph("Total Frame Valid", body_style), Paragraph(f"{len(valid_errs)} Frame", body_style)]
]

t = Table(tbl_data, colWidths=[280, 260])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#EAECEE")),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#BDC3C7")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE')
]))
story.append(t)
story.append(Spacer(1, 10))

if os.path.exists(chart_error_path):
    story.append(Paragraph("Grafik Trajektori Center Distance Error (px):", styles['Heading2']))
    story.append(RLImage(chart_error_path, width=540, height=225))

doc.build(story)

print(f"======================================================================")
print(f"✅ EVALUASI BENCHMARK LPW SELESAI!")
print(f"📁 Folder Output  : Output_LPW_{lpw_id}")
print(f"📄 File pred.txt   : pred.txt")
print(f"📄 File PDF Report: {os.path.basename(pdf_out_path)}")
print(f"======================================================================\n")


### <font color="#00c2cb" face="Palatino Linotype">**SOP-08: Penyimpanan & Tata Kelola Dataset (Private Research Storage)**</font>

**Deskripsi:**
Tata kelola penyimpanan dataset internal tim riset (Google Drive & Harddisk 1TB Privat) yang berstatus **Restricted / Private**.

### <font color="#00c2cb" face="Palatino Linotype">**SOP-09: Dokumentasi Riset & Publikasi Jurnal Scientific**</font>

**Deskripsi:**
Penyusunan naskah akademik manuskrip jurnal ilmiah dan proposal hibah penelitian berdasarkan luaran analisis otomasisasi 9-SOP.
